In [1]:
import os
import json
from unstructured.partition.pdf import partition_pdf

/Users/aniruddhajoshi/hrag/env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def parse_pdf_hierarchically(pdf_path):  
    """
    Reads a PDF, detects its visual layout, and groups paragraphs 
    under their respective section headers.
    """
    print(f" Processing: {os.path.basename(pdf_path)}... (This might take a minute on the first run)")
    
    # The 'hi_res' strategy uses vision models to detect layout boxes (Titles, Tables, etc.)
    try:
        elements = partition_pdf(
            filename=pdf_path, 
            strategy="hi_res",
            infer_bounding_boxes=True
        )
    except Exception as e:
        print(f"Error parsing {pdf_path}: {e}")
        return []

    document_chunks = []
    current_section = "Abstract / Introduction" # Default starting section
    
    for element in elements:
        element_type = element.category
        clean_text = str(element.text).replace("\n", " ").strip()
        
        # 1. Update our tracker if we hit a new Section Header or Title
        if element_type in ["Title", "Header"]:
            # Only update if the header is meaningful (not just a page number)
            if len(clean_text) > 3: 
                current_section = clean_text
                
        # 2. Extract Data Tables perfectly
        elif element_type == "Table":
            # Unstructured converts the table to HTML natively!
            table_html = element.metadata.text_as_html if hasattr(element.metadata, 'text_as_html') else clean_text
            document_chunks.append({
                "type": "table",
                "text": table_html,
                "metadata": {
                    "source": os.basename(pdf_path),
                    "section": current_section
                }
            })
            
        # 3. Extract Narrative Text and attach the current header
        elif element_type in ["NarrativeText", "Text", "ListItem"]:
            # Ignore tiny artifact strings (like isolated periods or single words)
            if len(clean_text) > 40:
                document_chunks.append({
                    "type": "text",
                    "text": clean_text,
                    "metadata": {
                        "source": os.basename(pdf_path),
                        "section": current_section
                    }
                })
                
    return document_chunks

In [7]:
# --- Test the Pipeline ---
papers_dir = "./dataset"

# Make sure the directory exists
if not os.path.exists(papers_dir):
    print(f"Could not find the '{papers_dir}' directory.")
    exit()

# Get the first PDF in the folder
pdf_files = [f for f in os.listdir(papers_dir) if f.endswith('.pdf')]

if not pdf_files:
    print(" No PDFs found in the 'papers' folder!")
    exit()
    
test_pdf = os.path.join(papers_dir, pdf_files[0])

# Run the parser
chunks = parse_pdf_hierarchically(test_pdf)

print(f"\n Successfully extracted {len(chunks)} contextual chunks!")
print("-" * 50)
print("Here is a sample of what your parsed data looks like:\n")

# Print a sample of the first 3 chunks to verify
for i, chunk in enumerate(chunks[5:8]): # Skipping first few in case of title pages
    print(json.dumps(chunk, indent=2))

No languages specified, defaulting to English.


 Processing: p1.pdf... (This might take a minute on the first run)


AttributeError: module 'os' has no attribute 'basename'